# Notebook 2: Scaffold Split & Feature Generation
**Input:** `data/CAlbicans_curated.csv`
**Output:** `data/train.csv`, `data/test.csv`, ECFP4 `.npy` arrays, descriptor CSVs, `data/split_log.json`


In [ ]:
import pandas as pd, numpy as np, os, json, random, warnings
warnings.filterwarnings('ignore')
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.DataStructs import ConvertToNumpyArray
from sklearn.model_selection import train_test_split
from collections import defaultdict
import matplotlib.pyplot as plt

SEED     = 42
DATA_DIR = 'data'
random.seed(SEED); np.random.seed(SEED)
df = pd.read_csv(os.path.join(DATA_DIR,'CAlbicans_curated.csv'))
print(f'Loaded: {len(df)} compounds | Active: {df["label"].sum()} ({df["label"].mean()*100:.1f}%)')

In [ ]:
# Stratified scaffold split
scaffold_summary = df.groupby('scaffold').agg(
    n_compounds=('label','count'), active_rate=('label','mean')
).reset_index()
scaffold_summary['scaffold_label'] = (scaffold_summary['active_rate']>=0.5).astype(int)

scaffolds_train, scaffolds_test = train_test_split(
    scaffold_summary['scaffold'], test_size=0.2,
    random_state=SEED, stratify=scaffold_summary['scaffold_label']
)
scaffolds_train, scaffolds_test = set(scaffolds_train), set(scaffolds_test)

df_train = df[df['scaffold'].isin(scaffolds_train)].reset_index(drop=True)
df_test  = df[df['scaffold'].isin(scaffolds_test)].reset_index(drop=True)
overlap  = scaffolds_train & scaffolds_test

print(f'Train: {len(df_train)} | Active: {df_train["label"].mean()*100:.1f}%')
print(f'Test:  {len(df_test)}  | Active: {df_test["label"].mean()*100:.1f}%')
print(f'Scaffold overlap: {len(overlap)} (must be 0)')

In [ ]:
# Nearest-neighbour Tanimoto (test -> train)
def get_fp(smi, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)

sample_test = df_test.sample(min(500,len(df_test)), random_state=SEED)
train_fps   = [get_fp(s) for s in df_train['std_smiles']]
train_fps   = [f for f in train_fps if f is not None]
nn_sims     = []
for smi in sample_test['std_smiles']:
    fp = get_fp(smi)
    if fp is None: continue
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    nn_sims.append(max(sims))
nn_arr = np.array(nn_sims)
print(f'Mean NN Tanimoto: {nn_arr.mean():.3f} | Median: {np.median(nn_arr):.3f}')

In [ ]:
# ECFP4 fingerprints
def smiles_to_ecfp4(smiles_list, radius=2, nbits=2048):
    out = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        arr = np.zeros(nbits, dtype=np.uint8)
        if mol:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)
            ConvertToNumpyArray(fp, arr)
        out.append(arr)
    return np.array(out)

X_train_fp = smiles_to_ecfp4(df_train['std_smiles'])
X_test_fp  = smiles_to_ecfp4(df_test['std_smiles'])
y_train    = df_train['label'].values
y_test     = df_test['label'].values
print(f'X_train_fp: {X_train_fp.shape} | X_test_fp: {X_test_fp.shape}')

In [ ]:
# RDKit 2D descriptors
desc_names = [d[0] for d in Descriptors.descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)

def calc_desc(smiles_list):
    rows = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        rows.append(list(calculator.CalcDescriptors(mol)) if mol else [np.nan]*len(desc_names))
    return pd.DataFrame(rows, columns=desc_names)

df_desc_train = calc_desc(df_train['std_smiles'])
df_desc_test  = calc_desc(df_test['std_smiles'])
nan_frac  = df_desc_train.isna().mean()
keep1     = nan_frac[nan_frac<0.1].index
df_desc_train = df_desc_train[keep1].fillna(df_desc_train[keep1].median())
df_desc_test  = df_desc_test[keep1].fillna(df_desc_train[keep1].median())
keep2         = df_desc_train.std()[df_desc_train.std()>0.01].index
df_desc_train = df_desc_train[keep2]
df_desc_test  = df_desc_test[keep2]
print(f'Descriptors retained: {df_desc_train.shape[1]}')

In [ ]:
# Save all outputs
df_train.to_csv(os.path.join(DATA_DIR,'train.csv'), index=False)
df_test.to_csv(os.path.join(DATA_DIR,'test.csv'),   index=False)
np.save(os.path.join(DATA_DIR,'X_train_ecfp4.npy'), X_train_fp)
np.save(os.path.join(DATA_DIR,'X_test_ecfp4.npy'),  X_test_fp)
np.save(os.path.join(DATA_DIR,'y_train.npy'), y_train)
np.save(os.path.join(DATA_DIR,'y_test.npy'),  y_test)
df_desc_train.to_csv(os.path.join(DATA_DIR,'X_train_desc.csv'), index=False)
df_desc_test.to_csv(os.path.join(DATA_DIR,'X_test_desc.csv'),   index=False)
split_log = {'train_compounds':len(df_train),'test_compounds':len(df_test),
             'train_active':int(y_train.sum()),'test_active':int(y_test.sum()),
             'train_active_rate':round(float(y_train.mean()),4),
             'test_active_rate':round(float(y_test.mean()),4),
             'train_scaffolds':len(scaffolds_train),'test_scaffolds':len(scaffolds_test),
             'scaffold_overlap':len(overlap),
             'nn_tanimoto_mean':round(float(nn_arr.mean()),4),
             'nn_tanimoto_median':round(float(np.median(nn_arr)),4),
             'ecfp4_bits':2048,'rdkit_descriptors':int(df_desc_train.shape[1]),'random_seed':SEED}
with open(os.path.join(DATA_DIR,'split_log.json'),'w') as f:
    json.dump(split_log, f, indent=2)
print(json.dumps(split_log, indent=2))